# Style response diagnostic: requested vs. achieved

For each of the 6 style axes we set that axis of the conditioning vector to a
range of **requested** values (-2σ … +2σ), decode the *same* content codes, then
run the decoded output back through the **same `StyleMeasurer`** the model was
trained against to read the **achieved** style value.

Plotting requested (x) vs achieved (y) gives, per axis, a curve whose **slope**
is the control gain:

* slope ≈ 1  → the knob works (output tracks the request),
* slope ≈ 0 & flat everywhere → the knob is ignored / the model can't represent it,
* positive near 0 but flattening at the extremes → control exists but **saturates**.

No video rendering — this runs in seconds.

In [ ]:
from pathlib import Path
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# Resolve project root robustly whether launched from repo root or main/.
cwd = Path.cwd()
if (cwd / 'config').exists():
    project_root = cwd
elif (cwd.parent / 'config').exists():
    project_root = cwd.parent
elif Path('/mnt/fastertalk/config').exists():
    project_root = Path('/mnt/fastertalk')
else:
    raise RuntimeError('Could not locate project root containing config/.')

os.chdir(project_root)
print('Project root:', Path.cwd())

from utils.config import load_flat_config
from models import get_model
from dataset.data_loader_joint_data_batched import get_dataloaders
from dataset.style_measure import StyleMeasurer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# --- Config, checkpoint, style axes ---
cfg = load_flat_config('config/talkinghead-1kh/stage1_style.yaml')
cfg.batch_size = 1
cfg.save_path = '/mnt/fastertalk/logs/stage1_style/checkpoints/epoch_300.pt'

# Style axes are read from the dataset stats so they stay in sync with what was
# annotated/trained (e.g. 'pose' appears only when it was included).
stats_path = Path(cfg.data_root) / 'annotations' / 'style_disp_stats.npz'
_stats = np.load(stats_path, allow_pickle=True)
_regions = [str(r) for r in _stats['region_names']]
_features = [str(f) for f in _stats['feature_names']] if 'feature_names' in _stats.files else ['disp', 'speed']
STYLE_NAMES = [f'{r}_{f}' for r in _regions for f in _features]
N_STYLE = len(STYLE_NAMES)

# Requested style values to probe along each axis (in z-scored σ units).
REQUESTED = np.array([-2.0, -1.0, 0.0, 1.0, 2.0], dtype=np.float32)

print('Checkpoint:', cfg.save_path)
print('Style axes:', STYLE_NAMES)
print('Requested values:', REQUESTED.tolist())


In [ ]:
# --- Load model, StyleMeasurer, and test data ---
dls = get_dataloaders(cfg)
test_loader = dls['test']

def _resolve_checkpoint(path_cfg):
    path_cfg = Path(path_cfg)
    if path_cfg.is_file():
        return path_cfg
    if path_cfg.is_dir():
        ckpts = sorted(path_cfg.glob('epoch_*.pt'))
        return ckpts[-1] if ckpts else None
    return None

ckpt_file = _resolve_checkpoint(cfg.save_path)
model = get_model(cfg).to(device)
if ckpt_file is not None:
    ckpt = torch.load(ckpt_file, map_location=device)
    state_dict = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print('Loaded checkpoint:', ckpt_file)
    print('Missing keys:', len(missing), '| Unexpected keys:', len(unexpected))
else:
    print('No valid checkpoint found - using random weights.')
model.eval()

measurer = StyleMeasurer(stats_path).to(device)
measurer.eval()
print('StyleMeasurer ready. stats:', stats_path, '| n_regions:', measurer.n_regions)
assert measurer.n_regions * 2 == N_STYLE, (measurer.n_regions, N_STYLE)


In [ ]:
# --- Collect a fixed set of test clips to average the response over ---
NUM_CLIPS = 8

clips = []
for i, batch in enumerate(test_loader):
    if i >= NUM_CLIPS:
        break
    blendshapes_in, blendshapes_tgt, mask, style = batch
    clips.append((blendshapes_in.to(device), mask.to(device)))
print(f'Collected {len(clips)} test clips.')

In [ ]:
# --- Core sweep: for each axis and each requested value, decode and measure ---
# achieved[axis, value, clip] = measured value of that axis after decoding with
# the conditioning vector set to `value` on `axis` (all other axes = 0).

@torch.no_grad()
def measure_response(model, measurer, clips):
    n_vals = len(REQUESTED)
    # achieved[axis, value] averaged over clips
    achieved = np.zeros((N_STYLE, n_vals), dtype=np.float32)

    for blendshapes_in, mask in clips:
        # Encode content once per clip (codes are shared across all style settings).
        quantized, _ = model.get_quant(blendshapes_in, mask)

        for axis in range(N_STYLE):
            for vi, val in enumerate(REQUESTED):
                style_vec = torch.zeros(1, N_STYLE, device=device)
                style_vec[0, axis] = float(val)
                pred = model.decode(quantized, mask, style=style_vec)   # [1, T, 58]
                meas = measurer(pred, mask)                             # [1, N_STYLE]
                achieved[axis, vi] += float(meas[0, axis].item())

    achieved /= len(clips)
    return achieved

achieved = measure_response(model, measurer, clips)
print('achieved shape:', achieved.shape, '(axis, value)')


In [ ]:
# --- Per-axis slope (control gain) via least-squares fit of achieved vs requested ---
def slope_intercept(x, y):
    A = np.vstack([x, np.ones_like(x)]).T
    m, b = np.linalg.lstsq(A, y, rcond=None)[0]
    return m, b

print(f"{'axis':<16} {'slope':>8} {'achieved @ -2..+2'}")
print('-' * 60)
slopes = {}
for axis, name in enumerate(STYLE_NAMES):
    m, b = slope_intercept(REQUESTED, achieved[axis])
    slopes[name] = m
    vals = '  '.join(f'{v:+.2f}' for v in achieved[axis])
    print(f'{name:<16} {m:>8.3f}   [{vals}]')

In [ ]:
# --- Plot requested vs achieved for all axes ---
ncols = 4
nrows = int(np.ceil(N_STYLE / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes_flat = np.atleast_1d(axes).flat
for axis, name in enumerate(STYLE_NAMES):
    ax = axes_flat[axis]
    ax.plot(REQUESTED, achieved[axis], 'o-', label='achieved')
    ax.plot(REQUESTED, REQUESTED, '--', color='gray', alpha=0.6, label='ideal (slope=1)')
    ax.set_title(f'{name}  (slope={slopes[name]:.2f})')
    ax.set_xlabel('requested (σ)')
    ax.set_ylabel('achieved (σ)')
    ax.axhline(0, color='k', lw=0.5, alpha=0.3)
    ax.axvline(0, color='k', lw=0.5, alpha=0.3)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
# Hide any unused subplots.
for j in range(N_STYLE, nrows * ncols):
    axes_flat[j].axis('off')
fig.suptitle('Style response: requested vs achieved (slope = control gain)', fontsize=14)
fig.tight_layout()
plt.show()


## How to read the results

* **slope ≈ 1, points on the dashed line** — the knob works (expect this for `lips_*`).
* **slope ≈ 0, flat line across all requested values** — the knob is ignored *and*
  the model cannot represent the change → measurement / leverage problem.
* **positive slope near 0 but the curve flattens at ±2σ** — control exists but
  **saturates** → code-preservation pinning or out-of-distribution extremes.

The flat vs. plateau distinction looks identical in a rendered video but points to
opposite fixes. To test the saturation hypothesis directly, retrain with a lower
`code_preservation_weight` (e.g. 0.02) and rerun this notebook: if the
forehead/eye slopes rise, code-preservation was the bottleneck.